# Stochastic Calculus from Zero · Episode 11: Quadratic variation, and what you can measure

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path is simulated with a fixed seed. The share is illustrative: 100, volatility 20%, real-world drift 8% a year.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. The multiplication table · 4. Scaled paths · 5. Volatility you can measure · 6. Drift you can't · 7. Sampling more often · 8. This episode's question · 9. Exercises · 10. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 11,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)

## 2. Last episode's answer

Halving $h$ four times divides it by 16, so the typical slope, about $1/\sqrt{h}$, grows by $\sqrt{16} = 4$.

In [ ]:
out["prev"] = {"answer": 4}

## 3. The multiplication table

On one fine path (the same $2^{20}$-step path as Episode 10), measured with coarser and finer grids, add up three things: the squared steps $\sum (\Delta W)^2$, the cross terms $\sum |\Delta W\,\Delta t|$, and $\sum (\Delta t)^2$. The first settles on the elapsed time; the other two vanish. That is the **quadratic variation** of Brownian motion, $[W]_t = t$, and the multiplication table Episode 15 runs on:

$$dW \cdot dW = dt, \qquad dW \cdot dt = 0, \qquad dt \cdot dt = 0 .$$

In [ ]:
rng = np.random.default_rng(seed)
brownian_paths(1, 1_000, 1.0, rng)                 # Episode 1's path; skip it
FINE = 2 ** 20
_, Wf, _ = brownian_paths(1, FINE, 1.0, rng)       # Episode 10's fine path
wf = Wf[0]
table = []
for n in (16, 256, 4_096, FINE):
    d = np.diff(wf[:: FINE // n])
    dt = 1 / n
    table.append({"steps": n, "dWdW": float((d ** 2).sum()), "dWdt": float(np.abs(d * dt).sum()), "dtdt": n * dt * dt})
assert abs(table[-1]["dWdW"] - 1) < 0.01 and table[-1]["dWdt"] < 1e-3
out["table"] = {"rows": table, "qv_fine": table[-1]["dWdW"], "qv_16": table[0]["dWdW"], "cross_fine": table[-1]["dWdt"]}
pd.DataFrame(table)

## 4. Scaled paths

For $\sigma W$, every squared step is multiplied by $\sigma^2$, so $[\sigma W]_t = \sigma^2 t$. A drift doesn't change it: $[\mu t + \sigma W]_t = \sigma^2 t$, because the drift's contributions are $dt\cdot dt$ and $dt \cdot dW$ terms.

In [ ]:
sig, mu = 0.20, 0.08
x = mu * np.linspace(0, 1, FINE + 1) + sig * wf
out["scaled"] = {"sigma": sig, "qv": float((np.diff(x) ** 2).sum()), "theory": sig ** 2}
assert abs(out["scaled"]["qv"] - sig ** 2) < 0.001
out["scaled"]

## 5. Volatility you can measure

A share with volatility $\sigma = 20\%$ and drift $\mu = 8\%$: its daily log returns are $(\mu - \tfrac12\sigma^2)\Delta t + \sigma \Delta W$. Their sum of squares over a year is the quadratic variation of the log price, $\sigma^2 \times 1$, so the **realised volatility** $\sqrt{\sum r_i^2}$ estimates $\sigma$. Over 10,000 simulated years it is right to within about one percentage point, with standard deviation close to $\sigma/\sqrt{2 \times 252}$.

In [ ]:
YEARS, DAYS = 10_000, 252
r = (mu - 0.5 * sig ** 2) / DAYS + sig * math.sqrt(1 / DAYS) * rng.standard_normal((YEARS, DAYS))
vol_hat = realised_vol(r, 1.0)
mu_hat = r.sum(axis=1) + 0.5 * vol_hat ** 2                 # the drift estimate from the same year
def hist(values, lo, hi, width):
    edges = np.arange(lo, hi + width / 2, width)
    counts, _ = np.histogram(values, bins=edges)
    return {"x": ((edges[:-1] + edges[1:]) / 2 * 100).round(3).tolist(), "y": (counts / len(values)).round(5).tolist(),
            "bar_width": width * 100, "x_range": [lo * 100, hi * 100]}
out["vol"] = {"years": YEARS, "mean": float(vol_hat.mean()), "sd": float(vol_hat.std()), "sd_theory": sig / math.sqrt(2 * DAYS),
              "hist": hist(vol_hat, 0.16, 0.24, 0.0025),
              "ticks": [{"x": v, "label": f"{v:.0f}%"} for v in (16, 18, 20, 22, 24)]}
assert abs(out["vol"]["sd"] / out["vol"]["sd_theory"] - 1) < 0.05
{k: v for k, v in out["vol"].items() if k in ("mean", "sd", "sd_theory")}

## 6. Drift you can't

The same year estimates the drift with standard deviation $\sigma/\sqrt{T} = 20\%$: useless. The total log return over a year is the only information about the drift, however finely you sample it. Ten years still leave $20\%/\sqrt{10} \approx 6.3\%$.

In [ ]:
r10 = (mu - 0.5 * sig ** 2) / DAYS + sig * math.sqrt(1 / DAYS) * rng.standard_normal((YEARS, 10 * DAYS))
mu10 = r10.sum(axis=1) / 10 + 0.5 * sig ** 2
out["drift"] = {"mean": float(mu_hat.mean()), "sd": float(mu_hat.std()), "sd_10y": float(mu10.std()),
                "hist": hist(mu_hat, -0.60, 0.76, 0.04), "true": mu,
                "ticks": [{"x": v, "label": f"{v:.0f}%"} for v in (-60, -40, -20, 0, 20, 40, 60)]}
assert abs(out["drift"]["sd"] / sig - 1) < 0.05 and abs(out["drift"]["sd_10y"] / (sig / math.sqrt(10)) - 1) < 0.05
{k: v for k, v in out["drift"].items() if k in ("mean", "sd", "sd_10y")}

## 7. Sampling more often

Observe the same year monthly, weekly, daily, or every quarter of an hour (26 times a trading day). The volatility estimate sharpens like $1/\sqrt{\text{observations}}$; the drift estimate doesn't move.

In [ ]:
freq = []
for label, n_obs in (("monthly", 12), ("weekly", 52), ("daily", 252), ("every 15 minutes", 252 * 26)):
    paths_ = 2_000
    rr = (mu - 0.5 * sig ** 2) / n_obs + sig * math.sqrt(1 / n_obs) * rng.standard_normal((paths_, n_obs))
    v = realised_vol(rr, 1.0)
    m = rr.sum(axis=1) + 0.5 * v ** 2
    freq.append({"sampling": label, "obs": n_obs, "vol_sd": float(v.std()), "drift_sd": float(m.std())})
assert freq[-1]["vol_sd"] < freq[0]["vol_sd"] / 10 and abs(freq[-1]["drift_sd"] / freq[0]["drift_sd"] - 1) < 0.15
out["freq"] = {"rows": freq}
pd.DataFrame(freq)

## 8. This episode's question

The quadratic variation of $3W$ over two years is $3^2 \times 2 = 18$.

In [ ]:
out["question"] = {"scale": 3, "years": 2, "answer": 9 * 2}

## 9. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** What is the quadratic variation of $3W$ over $[0, 2]$? Of $W_t + 5t$ over $[0, 1]$?

<details><summary>Answer</summary>

$9 \times 2 = 18$; and $1$: the drift adds nothing.

</details>

**Exercise 2 (By hand).** Why does $\sum |\Delta W\,\Delta t|$ go to zero?

<details><summary>Answer</summary>

$\sum |\Delta W|\,\Delta t \le \max|\Delta W| \cdot \sum \Delta t = \max|\Delta W| \cdot T$, and the largest step of a continuous path goes to 0.

</details>

**Exercise 3 (By hand).** With $\sigma = 20\%$, how many years of data give a drift estimate with standard deviation 2%?

<details><summary>Answer</summary>

$\sigma/\sqrt{T} = 2\%$ needs $\sqrt{T} = 10$: a hundred years.

</details>

**Exercise 4 (In code).** Compare the realised volatility with $\mu = 0$ and $\mu = 50\%$ on the same random draws, sampled daily and every 15 minutes. Why does the gap shrink?

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
gaps = {}
for n in (252, 252 * 26):        # daily, every 15 minutes
    z = rng.standard_normal((1_000, n))
    v0 = realised_vol((0 - 0.5 * sig ** 2) / n + sig * math.sqrt(1 / n) * z, 1.0)
    v5 = realised_vol((0.5 - 0.5 * sig ** 2) / n + sig * math.sqrt(1 / n) * z, 1.0)
    gaps[n] = float(v5.mean() - v0.mean())
assert gaps[252 * 26] < gaps[252] / 10       # the drift's squared steps are (mu dt)^2: they vanish as dt shrinks
print(f"drift adds {gaps[252]:.4f} to realised vol sampled daily, {gaps[252 * 26]:.5f} every 15 minutes")

**Exercise 5 (In code).** Estimate the quadratic variation of $W_t^2$ on the fine path. Compare it with $\int_0^1 4W_t^2\,dt$ (Episode 15 explains why they match).

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
qv_sq = float((np.diff(wf ** 2) ** 2).sum())
integral = float((4 * wf[:-1] ** 2).sum() / FINE)
assert abs(qv_sq / integral - 1) < 0.02
print(f"QV of W^2: {qv_sq:.4f}; integral of 4 W^2 dt: {integral:.4f}")

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")